# sources

> Add concepts to the concept graph from any text, from a vault document and from vedicreader.

This module adds concepts to the `learning` shelf. A model finds the core ideas in a text, and code merges them with the concepts on the shelf.
The vedicreader import reads the repo files and needs no model.
The tutor then asks about the concepts that this module adds.

In [ ]:
#| default_exp sources

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

These are the imports of the module.

In [ ]:
#| export
import json, os, re
from pathlib import Path
from fastcore.utils import *
from abhyasa.store import Learning, slugify, _union
from abhyasa.llm import ask_json
from abhyasa.prompts import register, prompt

## Extraction from text

The model lists the core ideas of a text. Each idea has a true, self-contained statement and its prereqs.
Code does the merge. When a slug or an alias matches a concept on the shelf, it is the same concept.
The lists of that concept get the new items, and its learner state does not change.

`EXTRACT_SP` and `extract_prompt` ask the model for the concepts. The model also gives the section, one quote, the figure, the ways that suit the idea, and the people and dates. `EXTRACT_SP` is a registered prompt, so you can change it on the Settings page.

In [ ]:
#| export
EXTRACT_SP = register('extract', 'Concept finder', 'How the tutor finds the ideas in a text that you add.', """You find the core ideas in a text, for a learner who will be tested on them.
- A concept is one idea a learner can be asked about: a definition, a rule, a method, a result, a distinction.
- `statement` is true and self-contained: it makes sense without the text. One to three sentences. Maths in $...$.
- Skip trivia, examples and names that are not ideas. Prefer 4-15 concepts for a page or two.
- `prereqs` names other concepts (from your list or from the known list) that a learner must know first.
- List the concepts in the order of the text.
- `section` is the title of the part of the text that holds the idea.
- `quote` is one short sentence of the text, copied exactly.
- `figure` is the link of one image of the text that shows the idea, copied exactly. Use "" when there is none.
- `ways` names the ways that suit the idea. Use only names from the list in the request.
- `moves` names the things that change in the idea, for example "weights". Use [] when nothing changes.
- `when` is the year of the idea, or null. `who` names the people of the idea.""")

def extract_prompt(text, subject='', known=()):
    from abhyasa.method import METHODS
    kn = ', '.join(known[:150])
    return ((f"Subject: {subject}\n" if subject else '') + (f"Concepts the learner already has: {kn}\n" if kn else '')
            + f"Ways: {', '.join(METHODS)}\n\nText:\n{text}\n\n"
            'Return JSON: {"concepts": [{"name": str, "statement": str, "aliases": [str], "prereqs": [str], "section": str, "quote": str, '
            '"figure": str, "ways": [str], "moves": [str], "when": int|null, "who": [str]}]}')

`_index` finds the slug for a name or an alias. `_windows` cuts a long text at paragraph breaks. `doc_pages` reads every chunk of a vault document in page order. It adds 1 to the page, because the vault counts pages from 0 and a reader counts from 1.

In [ ]:
#| export
def _index(lr):
    "Map each lower-case name, alias and slug on the shelf to its slug."
    out = {}
    for c in lr.concepts():
        for n in (c.slug, c.name, *(c.aliases or ())): out[n.lower()] = c.slug
    return out

def _windows(text, size=12000):
    "Cut a long text at paragraph breaks into windows of about `size` characters."
    out, cur = [], ''
    for p in re.split(r'\n\s*\n', text):
        if cur and len(cur) + len(p) > size: out.append(cur); cur = ''
        cur += p + '\n\n'
    return [*out, cur] if cur.strip() else out

`doc_pages` returns the chunks of a document.

In [ ]:
#| export
def doc_pages(lr:Learning, ref:str, shelf:str='store') -> AttrDict:
    "Every chunk of one vault document in page order, as `(page, text)` with the page a reader sees (from 1)."
    sh = lr.vault if shelf == 'store' else lr.vault.shelf(shelf)
    if (d := sh.doc(ref)) is None: raise ValueError(f'no document in the vault matching {ref!r}')
    did = d['id'].replace("'", "''")
    cs = sorted(sh.store(where=f"doc_id='{did}'", select='content, page, rowid as rowid'), key=lambda c: (-1 if c['page'] is None else c['page'], c['rowid']))
    return AttrDict(doc_id=d['id'], title=d['title'], source=d['source'], pages=[(None if c['page'] is None else c['page'] + 1, c['content']) for c in cs])

`locate` finds a quote in the pages. It folds white space and case, and it looks in all pages, so a window border does not lose a quote.

In [ ]:
#| export
def _fold(s):
    "`s` in lower case with each run of white space made one space, and the place in `s` of each character."
    out, at = [], []
    for i, ch in enumerate(s):
        if ch.isspace():
            if out and out[-1] != ' ': out.append(' '); at.append(i)
        else:
            for x in ch.lower(): out.append(x); at.append(i)
    return ''.join(out), at

def _locate(pages, quote):
    "`(index of the chunk, start, end)` of `quote` in `pages`, or None."
    q = _fold((quote or '').strip())[0]
    if len(q) < 12: return None
    for i, (_, t) in enumerate(pages):
        f, at = _fold(t)
        if (j := f.find(q)) >= 0: return i, at[j], at[j + len(q) - 1] + 1

def locate(pages:list, quote:str):
    "`(page, start, end)` of `quote` in the `(page, text)` list `pages`, with white space and case folded. None when it is not there or too short."
    r = _locate(pages, quote)
    return r and (pages[r[0]][0], r[1], r[2])

`figure_path` keeps a figure only when the file is in the assets folder of the document and is an image.

In [ ]:
#| export
IMG_SUFFIXES = ('.png', '.jpg', '.jpeg', '.gif', '.webp')

def figure_path(lr:Learning, source:str, link:str):
    "The path of the image `link` of the document `source`, from the assets folder. None when the file is not there, is outside that folder or is not an image."
    if not (source and link): return None
    root = lr.vault.assets().resolve()
    try: p = (lr.vault.assets(Path(source).stem)/str(link)).resolve()
    except (ValueError, OSError): return None
    if p.suffix.lower() in IMG_SUFFIXES and p.is_file() and p.is_relative_to(root): return str(p.relative_to(root))

`placement` gives the entry of a concept for one document.

In [ ]:
#| export
def placement(c, doc_id:str):
    "The newest provenance entry of the concept `c` with this `doc_id`, or None."
    return next((AttrDict(p) for p in reversed(c.get('provenance') or ()) if p.get('doc_id') == doc_id), None)

`extract` adds the concepts of a text. When it gets pages, it also keeps where each idea is in the text: the order, the section, the page and the passage. `_read` asks the model for each window. `_places` puts the ideas in the order of the text. `from_vault` does the same for one document of the vault.

In [ ]:
#| export
def _strs(x): return [s for s in x if isinstance(s, str)] if isinstance(x, list) else []

def _doc_source(lr, source):
    "The source path of the vault document that `source` names, or ''."
    if not (source and source.get('doc_id')): return ''
    sh = lr.vault if source.get('shelf', 'store') == 'store' else lr.vault.shelf(source['shelf'])
    return (sh.doc(source['doc_id']) or {}).get('source') or ''

def _read(lr, ask, text, subject, pages):
    "Ask for the concepts of each window of the text, and return `(slug, concept, prereq slugs)` in the order of the replies."
    idx, out = _index(lr), []
    for w in _windows('\n\n'.join(t for _, t in pages)):
        r = ask_json(ask, extract_prompt(w, subject, sorted({v for v in idx.values()})), prompt('extract'))
        cs = r.get('concepts', []) if isinstance(r, dict) else r
        slug_of = {}
        for c in cs:
            names = [c['name'], *(c.get('aliases') or ())]
            slug_of[c['name'].lower()] = next((idx[n.lower()] for n in names if n.lower() in idx), slugify(c['name']))
        for c in cs:
            s = slug_of[c['name'].lower()]
            out.append((s, c, [slug_of.get(p.lower()) or idx.get(p.lower()) for p in c.get('prereqs') or ()]))
        for c in cs: idx.update({n.lower(): slug_of[c['name'].lower()] for n in (c['name'], *(c.get('aliases') or ()))})
    return out

def _places(pages, recs):
    "The place of each concept in the text: a list of `(order, located)`. A concept that is not found follows the one before it."
    locs, key, keys = [_locate(pages, c.get('quote')) for _, c, _ in recs], (-1, -1), []
    for l in locs: key = l[:2] if l else key; keys.append(key)
    rank = {i: n for n, i in enumerate(sorted(range(len(recs)), key=keys.__getitem__))}
    return [(rank[i], l) for i, l in enumerate(locs)]

def extract(lr:Learning, ask, text:str='', subject:str='', source:dict=None, pages:list=None) -> L:
    "Add the core ideas of `text` (or of `pages`, a list of `(page, text)`) to the shelf as concepts, merged with the concepts there. Return the slugs that it changed."
    pgs = pages if pages is not None else [(None, text)]
    recs = _read(lr, ask, text, subject, pgs)
    places = _places(pgs, recs) if pages is not None else [(None, None)] * len(recs)
    src = _doc_source(lr, source) if pages is not None else ''
    touched = L()
    for (s, c, pre), (order, loc) in zip(recs, places):
        old, fig = lr.concept(s) or {}, figure_path(lr, src, c.get('figure'))
        moves = _union(old.get('moves'), _strs(c.get('moves')))
        meta = dict(ways=_union(old.get('ways'), _strs(c.get('ways'))), moves=moves, who=_union(old.get('who'), _strs(c.get('who'))),
                    when=c.get('when', old.get('when')), image=fig or old.get('image'), visual=bool(moves))
        prov = [source] if source else []
        if pages is not None and source is not None:
            t = pgs[loc[0]][1][loc[1]:loc[2]][:600] if loc else ''
            prov = [{**source, 'order': order, 'section': c.get('section') or '', 'page': pgs[loc[0]][0] if loc else None,
                     'span': [loc[1], loc[1] + len(t)] if loc else None, 'passage': t, 'figure': fig}]
        lr.put(c['name'], c.get('statement', ''), subject, slug=s, aliases=c.get('aliases') or (), prereqs=[p for p in pre if p], provenance=prov, **meta)
        touched.append(s)
    return touched.unique()

def _cut(pages, chars):
    "The `(page, text)` list `pages`, cut to `chars` characters in all; all of it when `chars` is None."
    if chars is None: return pages
    out = []
    for p, t in pages:
        if chars <= 0: break
        out.append((p, t[:chars])); chars -= len(t)
    return out

def from_vault(lr:Learning, ask, ref:str, subject:str='', shelf:str='store', chars:int=None) -> L:
    "Extract concepts from one document of a shelf of the vault, by doc id, source or title; from its first `chars` characters when `chars` is given."
    d = doc_pages(lr, ref, shelf)
    return extract(lr, ask, subject=subject, source=dict(src='vault', shelf=shelf, doc_id=d.doc_id, title=d.title), pages=_cut(d.pages, chars))

A fake model returns three concepts. One matches `Ranking` on the shelf by its alias `rank`.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
lr = temp_learning()
lr.put('Ranking', 'Order items by a score.', 'ml', aliases=['rank'])
fake = Fake({'Return JSON: {"concepts"': dict(concepts=[
    dict(name='Reciprocal rank fusion', statement='Sum 1/(k + rank) over lists.', aliases=['RRF'], prereqs=['rank', 'Ranked list']),
    dict(name='Ranked list', statement='Items in score order.', aliases=[], prereqs=[]),
    dict(name='Ranking', statement='Ordering by score.', aliases=['ordering'], prereqs=[])])})

`extract` returns the changed slugs. The prereqs become slugs, and the old concept gets the new alias and statement.

In [ ]:
test_eq(sorted(extract(lr, fake, 'some text', 'ml', dict(src='test'))), ['ranked-list', 'ranking', 'reciprocal-rank-fusion'])
test_eq(lr.concept('reciprocal-rank-fusion').prereqs, ['ranking', 'ranked-list'])
test_eq(lr.concept('ranking').aliases, ['rank', 'ordering'])
test_eq(lr.concept('ranking').statement, 'Ordering by score.')

`_windows` cuts at paragraph breaks.

In [ ]:
test_eq(len(_windows('a\n\n' * 10, size=5)), 5)

## Reading order and where each idea stands

When the text comes as pages, `extract` also keeps the order, the section, the page and the passage of each idea. These tests use a small paper in a temporary vault. The paper data is in `tests/walk_data.py`.

In [ ]:
#| hide
import sys; sys.path.insert(0, '../tests'); from walk_data import *
from abhyasa.prompts import PROMPTS

A fake model names three ideas of the paper. `from_vault` returns them in the order of the text.

In [ ]:
lr = temp_learning(); did = add_paper(lr)
ask = Fake({'Return JSON: {"concepts"': EXTRACT_REPLY})
slugs = from_vault(lr, ask, did)
test_eq(list(slugs), ['sequential-computation', 'softmax', 'scaling-by-the-root-of-dk'])

The placement of an idea is in its provenance. The page is the page a reader sees: the vault counts pages from 0, so the stored page 3 is page 4.

In [ ]:
p = placement(lr.concept('softmax'), did)
test_eq((p.order, p.section, p.page), (1, '3.2.1 Scaled Dot-Product Attention', 4))
assert p.passage.startswith('apply a softmax function')
test_eq(p.figure, 'attention/images/fig2.png')

The way fields, the picture and the people go on the concept. The prompt has a label for the Settings page.

In [ ]:
test_eq(lr.concept('softmax').ways, ['code', 'maths'])
test_eq(lr.concept('softmax').visual, True)
test_eq(lr.concept('softmax').image, 'attention/images/fig2.png')
test_eq(lr.concept('scaling-by-the-root-of-dk').who, ['Vaswani'])
test_eq(PROMPTS['extract'].label, 'Concept finder')

A second document that names Softmax keeps its own placement. The ways of the two documents merge.

In [ ]:
d2 = lr.vault.add([(0, 'Softmax gives weights.')], 'Notes', source='/tmp/notes.md', kind='file')['doc_id']
from_vault(lr, Fake({'Return JSON: {"concepts"': dict(concepts=[dict(name='Softmax', statement='', quote='Softmax gives weights.', section='', ways=['words'])])}), d2)
test_eq(placement(lr.concept('softmax'), did).page, 4)
test_eq(placement(lr.concept('softmax'), d2).page, 1)
test_eq(lr.concept('softmax').ways, ['code', 'maths', 'words'])

A quote that is not in the text gives no passage. A figure file that is not there gives no figure.

In [ ]:
bad = dict(concepts=[dict(name='Masking', statement='The decoder hides later positions.', quote='we mask out all values', figure='images/nope.png', section='3.2.3')])
lr2 = temp_learning(); d3 = add_paper(lr2); from_vault(lr2, Fake({'Return JSON: {"concepts"': bad}), d3)
m = placement(lr2.concept('masking'), d3)
test_eq((m.passage, m.span, m.figure), ('', None, None))

A long paper has 15 pages. The model sees the paper in windows, and the last idea still gets its page.

In [ ]:
long = [(i, f'{i + 1} Section {i + 1}\n\n' + f'Idea {i + 1} is stated here. ' * 140) for i in range(15)]
lr3 = temp_learning(); d4 = lr3.vault.add(long, 'Long paper', source='/tmp/long.pdf', kind='pdf')['doc_id']
def per_window(p, sp):
    ns = sorted({int(n) for n in re.findall(r'Idea (\d+) is stated', p)})
    return json.dumps(dict(concepts=[dict(name=f'Idea {n}', statement=f'Idea {n}.', quote=f'Idea {n} is stated here.', section=f'{n} Section {n}') for n in ns]))
from_vault(lr3, per_window, d4)
test_eq(placement(lr3.concept('idea-15'), d4).page, 15)

`locate` folds white space and case. It refuses a quote that is too short.

In [ ]:
pg = [(1, 'One two.\n\nThree   FOUR five six.'), (2, 'Seven eight nine ten.')]
test_eq(locate(pg, 'three four FIVE six'), (1, 10, 31))
test_eq(locate(pg, 'seven eight nine'), (2, 0, 16))
test_is(locate(pg, 'three'), None)
test_is(locate(pg, 'nothing of this is there'), None)

`figure_path` keeps a figure only when the file is there, stays under the assets folder and has an image suffix.

In [ ]:
test_eq(figure_path(lr, '/tmp/attention.pdf', 'images/fig2.png'), 'attention/images/fig2.png')
test_is(figure_path(lr, '/tmp/attention.pdf', '../../../etc/hosts'), None)
test_is(figure_path(lr, '/tmp/attention.pdf', '/etc/hosts'), None)
(lr.vault.assets('attention')/'images'/'x.txt').write_text('x')
test_is(figure_path(lr, '/tmp/attention.pdf', 'images/x.txt'), None)

## From a link

You can give a link: an arXiv id or a PDF link, a docs page, or a GitHub repo or file. `file_link` files its pages in the vault, and the walk reads them in order.
abhyasa fetches a link only when you send the form, and only the link that you give. A crawl stays in the folder of the link and stops at the page cap.

`LINK_MAX` is the most pages that one link can give. `LINK_BYTES` is the most characters of one page that the vault keeps, and `LINK_CHARS` is the most that the tutor reads.
`FETCH` sets the limits of each fetch: a time limit and a speed limit, so one page cannot be larger than 30 MB.
`link_folder` gives the folder of a link, `in_scope` tells if a link is in it, and `link_sel` is the CSS selector of the links that the crawl follows.

In [ ]:
#| export
import ipaddress, socket
from urllib.parse import urljoin, urlparse

LINK_MAX = 20            # the most pages of one link
LINK_BYTES = 400_000     # the most characters of one page that the vault keeps
LINK_CHARS = 100_000     # the most characters of one page that the tutor reads
FETCH = dict(timeout=30, max_recv_speed=1_000_000, retries=1)   # each fetch: at most 30 s at 1 MB/s, so at most 30 MB

def link_folder(url:str) -> str:
    "The scheme, host and folder of `url`: all of it up to the last `/` of its path."
    u = urlparse(url)
    return f"{u.scheme}://{u.netloc}{u.path[:u.path.rfind('/') + 1] or '/'}"

def in_scope(url:str, start:str) -> bool:
    "Whether `url`, without its query and fragment, is in the folder of `start`. A path with `..` is never in it."
    u = urlparse(url)
    return '..' not in u.path.split('/') and f"{u.scheme}://{u.netloc}{u.path}".startswith(link_folder(start))

def link_sel(start:str) -> str:
    "The CSS selector of the links into the folder of `start`: from the root of the site, in full, or relative. None has `..`."
    f, up = link_folder(start).replace('"', '%22'), ':not([href*=".."])'
    return (f'a[href^="{urlparse(f).path}"]{up}, a[href^="{f}"]{up}, '
            f'a[href]:not([href*=":"]):not([href^="/"]):not([href^="#"]){up}')

`link_kind` names the reader for a link, as `fossick.what_is` does. It takes only an arXiv id, or an http or https link to a host on the internet.
The app must not fetch its own pages or the pages of the local network, so `_local` refuses a local name, a name with no dot, and an address that is not on the internet.
`_addrs` looks up a host name. `file_link` uses it before it fetches, because a name on the internet can point to a local address.

These checks have a limit. abhyasa looks up the host one time. Then fossick looks it up again, and curl looks it up a third time when it connects.
A host that gives a different address each time can get past the first two checks. abhyasa cannot give curl the checked address through fossick.
curl refuses a redirect to a local address, but it does not check the first address again.

In [ ]:
#| export
ARXIV_ID = re.compile(r'\d{4}\.\d{4,5}(?:v\d+)?')
LOCAL_ENDS = ('.localhost', '.local', '.internal', '.lan', '.home.arpa')
NOT_LOCAL = 'abhyasa does not read links to this computer or to a local network.'

class LinkError(ValueError):
    "A link that abhyasa does not read. Its words are for the learner."

def _ip(host):
    "`host` as an IP address, also in the short forms that the system takes, such as `127.1`; None for a name."
    try: return ipaddress.ip_address(host)
    except ValueError: pass
    try: return ipaddress.ip_address(socket.inet_aton(host))
    except OSError: return None

def _local(host:str) -> bool:
    "Whether `host` is this computer or the local network: a local name, a name with no dot, or an address that is not on the internet."
    h = host.lower().rstrip('.')
    if h == 'localhost' or h.endswith(LOCAL_ENDS) or ('.' not in h and ':' not in h): return True
    ip = _ip(h)
    return ip is not None and (not ip.is_global or ip.is_multicast)

def _addrs(host:str) -> set:
    "The addresses that DNS gives for `host`; none when it does not find the host."
    try: return {a[4][0] for a in socket.getaddrinfo(host, None)}
    except OSError: return set()

def link_kind(url:str) -> str:
    "The reader for `url`, as `fossick.what_is` names it: only for an arXiv id, or an http or https link to a host on the internet. Else `LinkError`."
    url = url.strip()
    if ARXIV_ID.fullmatch(url): return 'arxiv'
    try: u = urlparse(url); host = u.hostname
    except ValueError: host = None
    if not host or u.scheme not in ('http', 'https'): raise LinkError('abhyasa reads only web links (http or https) and arXiv ids.')
    if _local(host): raise LinkError(NOT_LOCAL)
    from fossick import what_is   # here, so that the import of this module stays light
    if (kind := what_is(url)) == 'youtube': raise LinkError('abhyasa cannot make a walk from a video link.')
    return kind

`reading_order` puts the pages of a docs site in the order of its menu. The start page comes first. Then come the pages in the order of their first link on the start page, and then the rest.

In [ ]:
#| export
def _bare(u): return urlparse(u)._replace(query='', fragment='').geturl().rstrip('/')

def reading_order(docs:list, start_text:str) -> L:
    "The start page, then the pages in the order of their first link in `start_text`, then the rest in their given order."
    if not docs: return L()
    at = {}
    for h in re.findall(r'\]\(\s*<?([^)\s>]+)', start_text or ''): at.setdefault(_bare(urljoin(docs[0].url, h)), len(at))
    return L([docs[0], *sorted(docs[1:], key=lambda d: at.get(_bare(d.url), len(at)))])

`file_link` files a link with the reader that fits it, and returns the pages in reading order, at most the page cap.

| link | reader | what the walk uses |
|---|---|---|
| an arXiv id or link | `Vault.arxiv` (the `papers` shelf) | the paper |
| a PDF link | `Vault.pdf` | the paper |
| a GitHub repo | `Vault.github` | its prose files, README first |
| a GitHub file | `Vault.gh_file` | the file |
| any other web page | `Vault.crawl`, or `Vault.url` when the cap is 1 | the start page and the pages in its folder |

`read(how, target, **kw)` does the fetch. The default calls the `Vault` method `how` on the shelf of the link, after it looks up the host.
A web page uses plain HTTP only (`auto=False`). A page that only a browser can read gives no page, because a browser runs the scripts of the page.
A docs site can redirect its start page to another host or folder. For a web page, `file_link` first reads the start page alone, and takes the folder from the address of that page after the redirects.
It checks that host like the first one, and refuses a host on this computer or the local network. Then the crawl starts from that page.
A test gives a fake `read`, so the test does not use the network.

In [ ]:
#| export
READERS = dict(arxiv='arxiv', pdf='pdf', github='github', ghfile='gh_file')
PLAIN = dict(auto=False, **FETCH)   # a web page: plain HTTP only, so no browser and never your own Chrome

def _filed(r) -> list:
    "The documents in what a reader returned: one dict, a list of dicts, or a repo with its `docs`."
    rs = r['docs'] if isinstance(r, dict) and 'docs' in r else [r] if isinstance(r, dict) else r or []
    return [x for x in rs if isinstance(x, dict) and x.get('doc_id')]

def _clip(src):
    "Text, or `(page, text)` pairs, cut to `LINK_BYTES` characters."
    return src[:LINK_BYTES] if isinstance(src, str) else _cut(src, LINK_BYTES)

def _reader(lr:Learning, kind:str):
    "The default `read`: the `Vault` method `how` on a new copy of the shelf of `kind`, whose `add` keeps at most `LINK_BYTES` of a page."
    def read(how, target, **kw):
        sh = lr.vault.shelf(lr.vault.route(kind).name)   # a copy, so the shared vault does not change
        add = sh.add
        sh.add = lambda src, *a, **k: add(_clip(src), *a, **k)
        return getattr(sh, how)(target, **kw)
    return read

def file_link(lr:Learning, url:str, max_pages:int=8, read=None) -> L:
    "File the pages of `url` in the vault. Return `shelf`, `doc_id`, `title` and `url` of each page in the folder of `url` (after any redirect of the start page), in reading order, at most `max_pages`."
    url, cap = url.strip(), max(1, min(int(max_pages), LINK_MAX))
    kind = link_kind(url)
    how = READERS.get(kind) or ('url' if cap == 1 else 'crawl')
    if read is None:
        if (host := urlparse(url).hostname) and any(map(_local, _addrs(host))): raise LinkError(NOT_LOCAL)
        read = _reader(lr, kind)
    shelves, docs = [lr.vault.route(kind), lr.vault], L()   # a reader can file on the shelf of the kind or on the main shelf
    start, found = url, read is not None
    if how in ('url', 'crawl'):   # a docs site can redirect its start page to another host or folder: the folder is that of the page after the redirects
        got = _filed(read('url', url, **PLAIN))
        start = first(d['source'] for r in got if (d := first(s.doc(r['doc_id']) for s in shelves))) or url
        host = urlparse(start).hostname
        if urlparse(start).scheme not in ('http', 'https') or not host or _local(host) or (not found and any(map(_local, _addrs(host)))): raise LinkError(NOT_LOCAL)
    kw = dict(crawl=dict(max_pages=cap, follow_sel=link_sel(start), **PLAIN), pdf=FETCH, arxiv=FETCH, github=dict(code=False)).get(how, {})
    for r in _filed(got if how == 'url' else read(how, start, **kw)):
        sh, d = first(((s, s.doc(r['doc_id'])) for s in shelves), lambda x: x[1]) or (None, None)
        if not d or d['id'] in docs.attrgot('doc_id') or (how in ('url', 'crawl') and not in_scope(d['source'], start)): continue
        docs.append(AttrDict(shelf=sh.name, doc_id=d['id'], title=d['title'], url=d['source']))
    if how == 'github': docs = L(sorted(docs, key=lambda d: 'readme' not in Path(d.url).name.lower()))
    if not docs: raise LinkError('abhyasa could not read this link.')
    return reading_order(docs, '\n\n'.join(t for _, t in doc_pages(lr, docs[0].doc_id, docs[0].shelf).pages))[:cap]

`link_folder` gives the folder of a link. `in_scope` keeps a link in that folder. It refuses a link on a host that only starts with the same name, and a path with `..`.

In [ ]:
START = 'https://docs.example.dev/en/docs/tool/overview'
test_eq(link_folder(START), 'https://docs.example.dev/en/docs/tool/')
test_eq(link_folder('https://ai.pydantic.dev'), 'https://ai.pydantic.dev/')
assert in_scope('https://docs.example.dev/en/docs/tool/setup#a', START) and not in_scope('https://docs.example.dev/en/api/x', START)
assert not in_scope('https://evil.example/en/docs/tool/x', START) and not in_scope('https://docs.example.dev.evil.example/en/docs/tool/', START)
assert not in_scope('https://docs.example.dev/en/docs/tool/../../admin', START)

`link_sel` is the CSS selector of the links that the crawl follows: a link from the root of the site into the folder, a full link into the folder, and a relative link.
No part takes a link with `..`, so a link cannot go up out of the folder. The crawl also stays on the same host, and `in_scope` checks each page after the crawl.

In [ ]:
from fossick.core import Selector
html = ''.join(f'<a href="{h}">x</a>' for h in ('agents/', '../models/', '/en/docs/tool/setup', 'https://docs.example.dev/en/docs/tool/hooks',
                                                  'https://evil.example/x', '#top', '/blog/news', '/en/docs/tool/../../admin', 'mailto:a@b.dev'))
test_eq([a.attrib['href'] for a in Selector(html).css(link_sel(START))],
        ['agents/', '/en/docs/tool/setup', 'https://docs.example.dev/en/docs/tool/hooks'])   # a relative link is followed; `..`, other hosts and other folders are not

`link_kind` takes only an arXiv id, or an http or https link to a host on the internet. It refuses a file, this computer and the local network.
It also refuses a video link. Each refusal is a `LinkError`, so the app can show its words to you.

In [ ]:
test_eq(link_kind('1706.03762'), 'arxiv')
test_eq(link_kind(START), 'web')
test_eq(link_kind('https://github.com/AnswerDotAI/fastcore'), 'github')
for bad in ('file:///etc/passwd', '/etc/passwd', 'ftp://example.com/a', 'http://localhost:5001/', 'http://127.0.0.1/', 'http://127.1/',
            'http://[::1]/', 'http://10.0.0.5/x', 'http://192.168.1.1/', 'http://169.254.169.254/latest', 'http://router/', 'http://nas.local/'):
    test_fail(lambda: link_kind(bad), exc=LinkError)
test_fail(lambda: link_kind('https://www.youtube.com/watch?v=x'), contains='video', exc=LinkError)

`reading_order` puts the start page first, then the pages in the order of their first link on the start page. A page with no link stays at the end.

In [ ]:
pg = lambda u: AttrDict(url=f'https://d.dev/docs/{u}')
test_eq(reading_order([pg('a'), pg('x'), pg('c'), pg('b')], 'See [B](b#top), [C](/docs/c) and [B again](b).').attrgot('url'),
        [pg(u).url for u in 'abcx'])

A fake reader takes the place of the network. It files four pages: the start page, two pages in its folder, and one blog page outside it.

In [ ]:
SITE = {START: '# Overview\n\nSee [Setup](/en/docs/tool/setup) and [Hooks](/en/docs/tool/hooks).',
        'https://docs.example.dev/en/docs/tool/hooks': '# Hooks\n\nA hook runs a command at a set point.',
        'https://docs.example.dev/en/docs/tool/setup': '# Setup\n\nInstall the tool with one command.',
        'https://docs.example.dev/blog/news': '# News\n\nWe have a new logo.'}
calls = []
def fake_read(lr):
    "File every page of SITE in crawl order, and keep the call."
    def read(how, target, **kw):
        calls.append((how, target, kw))
        return [lr.vault.add(md, md.split('\n')[0][2:], source=u, kind='web', meta=dict(url=u)) for u, md in SITE.items()]
    return read

The crawl gets the folder and the page cap, and the walk does not use the blog page. A cap of 1 reads one page. An arXiv id uses the arXiv reader. A link with no pages gives a clear message.

In [ ]:
lr = temp_learning()
docs = file_link(lr, START, max_pages=500, read=fake_read(lr))
crawl = first(c for c in calls if c[0] == 'crawl')
test_eq((calls[0][0], crawl[2]['max_pages']), ('url', LINK_MAX))   # the start page first, then the crawl
assert 'a[href^="/en/docs/tool/"]' in crawl[2]['follow_sel']
test_eq([d.title for d in docs], ['Overview', 'Setup', 'Hooks'])        # the start page, then its links in order; no blog page
test_eq(docs[0].shelf, 'store')
calls.clear(); test_eq(len(file_link(lr, START, max_pages=1, read=fake_read(lr))), 1); test_eq(calls[0][0], 'url')
calls.clear(); file_link(lr, '1706.03762', read=fake_read(lr)); test_eq(calls[0][0], 'arxiv')
test_fail(lambda: file_link(temp_learning(), START, read=lambda how, target, **kw: []), contains='could not read')

Before it fetches, `file_link` looks up the host. A host name that points to this computer or to the local network is refused. Here a fake lookup gives the address.

In [ ]:
was, _addrs = _addrs, lambda host: {'127.0.0.1'}
test_fail(lambda: file_link(temp_learning(), 'https://evil.example/docs/'), contains='local network')
_addrs = was

A docs site can redirect its start page to another host, or to another host and folder. `file_link` takes the folder from the page after the redirects. It refuses a redirect to a local host.


In [ ]:
REDIRECTS = {'https://old.example.dev/': 'https://new.example.dev/docs/ai/overview/',
             'https://docs.example.dev/en/docs/tool/overview': 'https://code.example.dev/docs/en/overview'}
def redirect_read(lr, to):
    "A fake `read` for a site that sends its start page to `to`; the pages of the site are in the folder of `to`."
    base = to.rsplit('/', 1)[0] + '/'
    site = {to: '# Overview\n\nSee [Setup](setup).', base + 'setup': '# Setup\n\nInstall it.', 'https://elsewhere.example/x': '# Other\n\nNot in the folder.'}
    def read(how, target, **kw):
        calls.append((how, target, kw))
        return [lr.vault.add(md, md.split('\n')[0][2:], source=u, kind='web') for u, md in (list(site.items())[:1] if how == 'url' else site.items())]
    return read
for start, to in REDIRECTS.items():
    calls.clear(); lr = temp_learning()
    test_eq([d.title for d in file_link(lr, start, 8, read=redirect_read(lr, to))], ['Overview', 'Setup'])   # the pages of the new folder
    test_eq(calls[-1][1], to)                                                                              # the crawl starts at the page after the redirect
    assert urlparse(to).path.rsplit('/', 1)[0] + '/' in calls[-1][2]['follow_sel']
    test_eq(len(file_link(lr, start, 1, read=redirect_read(lr, to))), 1)
for bad in ('http://127.0.0.1:8080/docs/x', 'http://intranet/docs/x'):
    lr = temp_learning()
    test_fail(lambda: file_link(lr, 'https://old.example.dev/', read=redirect_read(lr, bad)), contains='local network')

`_cut` keeps the first characters of the pages, for the cap on one linked page.

In [ ]:
test_eq(_cut([(1, 'abc'), (2, 'defg'), (3, 'h')], 5), [(1, 'abc'), (2, 'de')])
test_eq(_cut([(1, 'abc')], None), [(1, 'abc')])

The default reader uses plain HTTP only. It never starts a browser and never uses your own Chrome. Each fetch has a time limit and a speed limit, so one page cannot be very large.
The vault keeps at most `LINK_BYTES` characters of a page. Here fake `Vault` readers keep their keywords and file one page of 500 kB.

In [ ]:
from vishalakshi.core import Vault
READS = ('url', 'crawl', 'pdf', 'arxiv', 'gh_file', 'github')
got, real = {}, {k: getattr(Vault, k) for k in READS}
BIG = '# Big\n\n' + 'word ' * 100_000
def fake_vault(how):
    "A `Vault` reader that keeps its keywords and files one big page."
    def f(self, target, **kw):
        got[how] = kw
        r = self.add(BIG, 'Big', source=target if how in ('url', 'crawl') else f'{target}#{how}', kind='web')
        return L([r]) if how == 'crawl' else AttrDict(docs=L([r])) if how == 'github' else r
    return f
for k in READS: setattr(Vault, k, fake_vault(k))
was, _addrs = _addrs, lambda host: {'93.184.216.34'}
try:
    lr = temp_learning()
    for u, n in [(START, 1), (START, 8), ('https://x.example/paper.pdf', 8), ('1706.03762', 8),
                 ('https://github.com/a/b/blob/main/README.md', 8), ('https://github.com/a/b', 8)]: file_link(lr, u, n)
    test_eq(sorted(got), sorted(READS))
    test_eq((got['url']['auto'], got['crawl']['auto']), (False, False))           # plain HTTP: no browser, no logged-in Chrome
    assert not any(kw.get(k) for kw in got.values() for k in ('heavy', 'stealthy', 'session'))
    assert all(got[k]['max_recv_speed'] == FETCH['max_recv_speed'] and got[k]['timeout'] == FETCH['timeout'] for k in ('url', 'crawl', 'pdf', 'arxiv'))
    kept = sum(len(t) for _, t in doc_pages(lr, file_link(lr, START, 1)[0].doc_id).pages)
    assert kept <= LINK_BYTES * 1.05, kept                                          # the big page is cut before the vault keeps it
finally:
    for k, f in real.items(): setattr(Vault, k, f)
    _addrs = was

## vedicreader

The import reads the files of the vedicreader repo. It does not read the website. `$VEDICREADER_ROOT` gives the path of the repo.

| source | concepts |
|---|---|
| Sanskrit lessons | One concept for each lesson. The lesson before it in the course is its prereq. Its `check` pairs become ready questions at depth 4 (apply). Its recap becomes the statement. |
| `static/grammar/terms/*.json` | One concept for each term, with `related` links. A term with a jyotiṣa term of the same name also links to it. |
| `vr/jyotisha/explain/*.json` | One concept for each graha, rāśi and nakṣatra. |

`VR_ROOT` is the repo path. `plain` removes the marks of vedicreader from a text.

In [ ]:
#| export
VR_ROOT = Path(os.getenv('VEDICREADER_ROOT', Path.home()/'code/personal/orgs/vedicreader'))

def plain(s:str) -> str:
    "The vedicreader text without its `{…}` marks and `[1,2]` citations."
    return re.sub(r'\[\d+(?:,\s*\d+)*\]', '', re.sub(r'\{([^{}]*)\}', r'\1', s or '')).strip()

Three generators read one source each. Each generator yields the concepts as dicts for `put`. `_lesson_concepts` reads the lessons.

In [ ]:
#| export
def _lesson_concepts(root):
    d = root/'static/learn'
    order, prev = json.loads((d/'order.json').read_text()), None
    for ui, u in enumerate(order['units']):
        for li, ls in enumerate(u['lessons']):
            p = d/'lessons'/f"{ls['slug']}.json"
            body = json.loads(p.read_text()).get('body', []) if p.exists() else []
            recap = [plain(x) for b in body for x in b.get('recap', [])]
            qs = [dict(q=plain(c['q']), a=plain(c['a']), depth=4) for b in body for c in b.get('check', [])]
            yield dict(name=ls['title'], statement=' '.join([plain(ls['cap']), *recap]), subject='sanskrit', slug=f"sa-{ls['slug']}",
                       prereqs=[prev] if prev else [], questions=qs, order=ui*100 + li, unit=u['name'],
                       provenance=[dict(src='vedicreader', lesson=ls['slug'], url=f"https://vedicreader.com/v/learn/{ls['slug']}")])
            prev = f"sa-{ls['slug']}"

`_term_concepts` reads the grammar terms.

In [ ]:
#| export
def _term_concepts(root):
    for p in sorted((root/'static/grammar/terms').glob('*.json')):
        for t in json.loads(p.read_text()).get('terms', []):
            en = ', '.join(t.get('en') or [])
            yield dict(name=t.get('iast') or t['slug'], statement=plain(t.get('defn', '')), subject='sanskrit', slug=f"sa-term-{t['slug']}",
                       aliases=[a for a in (t.get('sa'), *(t.get('en') or ())) if a], group=t.get('group', ''), order=1000,
                       related=[f"sa-term-{r}" for r in t.get('related') or ()] + ([f"jy-term-{t['jy']}"] if t.get('jy') else []),
                       provenance=[dict(src='vedicreader', term=t['slug'], cite=(t.get('src') or {}).get('cite', ''))],
                       questions=[dict(q=f"What is {t.get('iast') or t['slug']} ({en})? Say what it is and where it is used.",
                                       a=plain(t.get('defn', '')), depth=2)] if t.get('defn') else [])

`_explain_concepts` reads the jyotiṣa explanations. The rāśi file keys each sign by its number, so `RASI` gives the names.

In [ ]:
#| export
RASI = [('Meṣa', 'Aries'), ('Vṛṣabha', 'Taurus'), ('Mithuna', 'Gemini'), ('Karkaṭa', 'Cancer'), ('Siṃha', 'Leo'), ('Kanyā', 'Virgo'),
        ('Tulā', 'Libra'), ('Vṛścika', 'Scorpio'), ('Dhanus', 'Sagittarius'), ('Makara', 'Capricorn'), ('Kumbha', 'Aquarius'), ('Mīna', 'Pisces')]

def _explain_name(kind, key):
    "The name and aliases of one explained item; the rāśi file keys each sign by its number, 0 for Meṣa."
    if kind == 'rasi' and key.isdigit(): nm, en = RASI[int(key)]; return nm, [en, 'rāśi', 'sign']
    return key, [kind]

def _explain_concepts(root):
    for kind in ('graha', 'rasi', 'nakshatra'):
        p = root/f'vr/jyotisha/explain/{kind}.json'
        if not p.exists(): continue
        for name, v in json.loads(p.read_text()).items():
            secs = v.get('sections', {})
            first_sec = next(iter(secs.values()), [])
            st = plain(first_sec[0] if first_sec else '')[:900]
            nm, al = _explain_name(kind, name)
            yield dict(name=nm, statement=st, subject='jyotisha', slug=f"jy-{kind}-{slugify(nm)}", kind=kind, order=2000,
                       aliases=al, provenance=[dict(src='vedicreader', explain=kind, key=name)])

`from_vedicreader` runs the generators that `parts` names and returns a count for each part.

In [ ]:
#| export
def from_vedicreader(lr:Learning, root=None, parts=('lessons', 'terms', 'explain')) -> dict:
    "Add the lessons, terms and jyotiṣa explanations of vedicreader as concepts. Keep the learner state on the shelf."
    root, n = Path(root or VR_ROOT), {}
    gens = dict(lessons=_lesson_concepts, terms=_term_concepts, explain=_explain_concepts)
    for part in parts:
        n[part] = 0
        for c in gens[part](root):
            lr.put(**c); n[part] += 1
    return n

`plain` removes the marks and the citations.

In [ ]:
test_eq(plain('{a} + {a} → {ā}: {nāsti}.[1,2]'), 'a + a → ā: nāsti.')

When the repo is on this machine, the lessons become a chain of concepts with prereqs and ready questions.

In [ ]:
if VR_ROOT.exists():
    lr = temp_learning()
    n = from_vedicreader(lr, parts=('lessons',))
    assert n['lessons'] > 10
    c = lr.concept('sa-sandhi-vowels')
    assert c.prereqs and c.questions and c.subject == 'sanskrit'
    test_eq(lr.concept('sa-sounds').prereqs, [])
    test_eq(_explain_name('rasi', '0'), ('Meṣa', ['Aries', 'rāśi', 'sign']))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()